In [7]:
import pandas as pd

# ----- Parameters and file paths -----
n_all = [3, 5, 7, 10]  # general number, e.g. 3, 5, or 7
# Variables (adjust these as needed)
Nbins = 2
num_top = 843
folder = 'Feature_Importance'
property = 'Activity'
importance_style = 'SHAP'
featurizer_name = 'RDKit_Descriptors_NG'
sheet_name = featurizer_name

importance_output_file = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{folder}\{property}_{Nbins} bins_{importance_style}_Importance.xlsx"
interactions_output_file = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{folder}\{property}_{Nbins} bins_{importance_style}_Top{num_top}_Interactions.xlsx"

# ----- Step 1: Load and process importance data -----
importance_df = pd.read_excel(importance_output_file, sheet_name=sheet_name)

# Sort by SHAP Importance (descending)
importance_df = importance_df.sort_values(by='SHAP Importance', ascending=False)

for n in n_all:

    # Get the top n features based on SHAP Importance (we use Feature ID for identification)
    top_features = importance_df.iloc[19*n:20*n]['Feature ID'].tolist()
    
    # ----- Step 2: Load interactions data -----
    interactions_df = pd.read_excel(interactions_output_file, sheet_name=sheet_name)
    
    # (Optional) Filter out problematic rows if needed:
    interactions_df = interactions_df[
        interactions_df['Interaction Importance'].notnull() &
        (interactions_df['Interaction Importance'] != 0)
    ]
    
    # ----- Step 3: For each top feature, get the top n interacting features -----
    # We'll collect features in a set (starting with the top features themselves).
    first_n_features = set(top_features)
    
    # For each feature in the top n features:
    for feat in top_features:
        # Filter rows where the top feature is involved (could be in either column)
        df_feat = interactions_df[
            (interactions_df['Feature 1 ID'] == feat) | (interactions_df['Feature 2 ID'] == feat)
        ]
        # Sort these interactions in descending order by Interaction Importance.
        df_feat = df_feat.sort_values(by='Interaction Importance', ascending=False)
        
        # For each row, determine the "other" feature (the one interacting with feat)
        interacting_list = []
        for _, row in df_feat.iterrows():
            if row['Feature 1 ID'] == feat:
                interacting_feature = row['Feature 2 ID']
            else:
                interacting_feature = row['Feature 1 ID']
            interacting_list.append(interacting_feature)
    
        # Remove duplicates while preserving order:
        unique_interactors = []
        for f in interacting_list:
            if f not in unique_interactors:
                unique_interactors.append(f)
            if len(unique_interactors) >= n:
                break  # only take top n interactors
        
        # Add these to our overall set.
        first_n_features.update(unique_interactors)
    
    # Convert the set to a list. (Optionally, sort it, e.g., by numeric value or by importance.)
    First_n = sorted(list(first_n_features))
    print(First_n)


[6, 11, 14, 18, 22, 45, 53, 59, 66, 67, 106, 236]
[11, 14, 17, 18, 20, 21, 24, 26, 51, 59, 63, 66, 73, 76, 82, 86, 98, 99, 115, 761]
[0, 1, 3, 4, 6, 13, 14, 15, 20, 21, 22, 23, 27, 31, 38, 43, 44, 46, 47, 48, 50, 53, 61, 63, 64, 65, 66, 74, 82, 83, 87, 88, 91, 94, 96, 97, 98, 99, 100, 104, 114, 139, 164, 198, 207, 230, 234, 257, 311, 331, 538]
[0, 1, 4, 5, 6, 8, 12, 13, 14, 18, 19, 20, 23, 24, 28, 30, 31, 32, 33, 37, 48, 53, 61, 65, 74, 83, 84, 87, 96, 97, 99, 105, 106, 107, 110, 116, 117, 122, 125, 126, 175, 207, 210, 225, 251, 301, 308, 316, 344, 374, 450, 459, 643, 842]
